# 1 - NLP Section
### Splitting the uploaded document into tokens and assigning relevant column values to each based on spaCy token attributes.

a) Installing necessary libraries.

In [ ]:
!pip install pymupdf spacy pandas
!python -m spacy download en_core_web_sm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.9/68.9 MB 41.5 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 97.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [pymupdf]
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 28.4 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


b) Extracting tokens from the pdf and assigning various attribute values using spaCy.

Revision: Ensuring pdf_layout.json also contains important text information such as font family, sizes, formatting etc to reconstruct the final document as close to the original as possible.

In [ ]:
import fitz
import spacy
import pandas as pd
import json
from pathlib import Path

nlp = spacy.load("en_core_web_sm")


def get_word_formatting(word_bbox, text_spans):
    """Find the text span that best matches a PDF word's bounding box."""

    wx0, wy0, wx1, wy1 = word_bbox
    best_span = None
    best_overlap = 0

    for span in text_spans:
        sx0, sy0, sx1, sy1 = span["bbox"]

        # Calculate intersection between word and span bounding boxes
        ix0 = max(wx0, sx0)
        iy0 = max(wy0, sy0)
        ix1 = min(wx1, sx1)
        iy1 = min(wy1, sy1)

        overlap = max(0, ix1 - ix0) * max(0, iy1 - iy0)

        if overlap > best_overlap:
            best_overlap = overlap
            best_span = span

    if best_span is None:
        return {
            "font": None,
            "font_size": None,
            "color": None,
            "color_hex": None,
            "flags": None,
            "bold": None,
            "italic": None
        }

    color = best_span.get("color")

    color_hex = (
        f"#{color:06X}"
        if color is not None
        else None
    )

    flags = best_span.get("flags", 0)

    return {
        "font": best_span.get("font"),
        "font_size": best_span.get("size"),
        "color": color,
        "color_hex": color_hex,
        "flags": flags,
        "bold": bool(flags & 16),
        "italic": bool(flags & 2)
    }


def process_pdf(pdf_path, output_dir="data/processed"):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    all_tokens = []
    all_pages = []

    with fitz.open(pdf_path) as pdf:
        for page_number, page in enumerate(pdf, start=1):
            pdf_words = page.get_text("words", sort=True)
            print(
                f"Page {page_number}: "
                f"{len(pdf_words)} extracted words"
            )

            if not pdf_words:
                print(
                    f"WARNING: Page {page_number} has no "
                    "extractable text. OCR may be required."
                )

            page_dict = page.get_text("dict")
            text_spans = []

            for block in page_dict["blocks"]:
                if "lines" not in block:
                    continue

                for line in block["lines"]:
                    for span in line["spans"]:
                        text_spans.append(span)

            word_records = []
            text_parts = []
            cursor = 0
            previous_block = None
            previous_line = None

            for word_data in pdf_words:
                x0, y0, x1, y1, word = word_data[:5]
                block_no, line_no, word_no = word_data[5:8]

                if not word:
                    continue

                if text_parts:
                    if block_no != previous_block:
                        separator = "\n\n"
                    elif line_no != previous_line:
                        separator = "\n"
                    else:
                        separator = " "

                    text_parts.append(separator)
                    cursor += len(separator)

                start_offset = cursor
                text_parts.append(word)
                cursor += len(word)

                formatting = get_word_formatting(
                    [x0, y0, x1, y1],
                    text_spans
                )

                word_records.append({
                    "word": word,
                    "start_offset": start_offset,
                    "end_offset": cursor,
                    "bbox": [x0, y0, x1, y1],
                    "block_no": block_no,
                    "line_no": line_no,
                    "word_no": word_no,

                    # Original text formatting
                    **formatting
                })

                previous_block = block_no
                previous_line = line_no

            page_text = "".join(text_parts)
            doc = nlp(page_text)

            for token in doc:
                if token.is_space:
                    continue

                overlapping = [
                    w for w in word_records
                    if w["start_offset"] < token.idx + len(token.text)
                    and w["end_offset"] > token.idx
                ]

                if overlapping:
                    x0 = min(w["bbox"][0] for w in overlapping)
                    y0 = min(w["bbox"][1] for w in overlapping)
                    x1 = max(w["bbox"][2] for w in overlapping)
                    y1 = max(w["bbox"][3] for w in overlapping)
                    block_no = overlapping[0]["block_no"]
                    line_no = overlapping[0]["line_no"]
                    word_no = overlapping[0]["word_no"]
                else:
                    x0 = y0 = x1 = y1 = None
                    block_no = line_no = word_no = None

                all_tokens.append({
                    "token_id": f"P{page_number:03d}_T{len(all_tokens):05d}",
                    "page_number": page_number,
                    "token": token.text,
                    "base_word": token.lemma_.lower(),
                    "normalized_word": token.text.lower(),
                    "lemma_reason": (
                        f"Changed from '{token.text}' to '{token.lemma_}'"
                        if token.text.lower() != token.lemma_.lower()
                        else "No change; token is already in its lemma form"
                    ),
                    "is_word": token.is_alpha,
                    "is_stopword": token.is_stop,
                    "is_num": token.like_num,
                    "is_punct": token.is_punct,
                    "is_currency": token.is_currency,
                    "is_url": token.like_url,
                    "is_email": token.like_email,
                    "start_offset": token.idx,
                    "end_offset": token.idx + len(token.text),
                    "x0": x0,
                    "y0": y0,
                    "x1": x1,
                    "y1": y1,
                    "block_no": block_no,
                    "line_no": line_no,
                    "word_no": word_no
                })

            all_pages.append({
                "page_number": page_number,
                "page_width": page.rect.width,
                "page_height": page.rect.height,
                "text": page_text,
                "words": word_records
            })

    tokens_df = pd.DataFrame(all_tokens)
    tokens_df["sensitivity"] = ""

    tokens_df.to_csv(output_dir / "tokens.csv", index=False)

    with open(output_dir / "pdf_layout.json", "w", encoding="utf-8") as file:
        json.dump(all_pages, file, ensure_ascii=False, indent=2)

    print(f"Pages processed: {len(all_pages)}")
    print(f"Tokens extracted: {len(tokens_df)}")
    print(f"Files saved in: {output_dir}")

    print(
        tokens_df[
            ["token", "base_word", "start_offset", "end_offset"]
        ].head(10).to_string(index=False)
    )

    return tokens_df, all_pages

if __name__ == "__main__":
    process_pdf("/content/Educational_Data_Document.pdf")

Page 1: 222 extracted words
Page 2: 209 extracted words
Page 3: 154 extracted words
Pages processed: 3
Tokens extracted: 657
Files saved in: data/processed
     token  base_word  start_offset  end_offset
 WESTFIELD  westfield             0           9
 INSTITUTE  institute            10          19
        OF         of            20          22
TECHNOLOGY technology            23          33
    Office     office            35          41
     ofthe      ofthe            42          47
University university            48          58
 Registrar  registrar            59          68
         &          &            69          70
  Academic   academic            71          79


Testing why same word maps to different base words based on POS (Parts of Speech) Rules.

In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm")

# Let's test "banking" in a noun context vs. a verb context
test_cases = [
    "Banking is a major global sector.",  # Noun/Gerund context
    "The banking industry is growing.",   # Adjectival/Noun modifier context
    "He is banking his checks today.",      # Verb context
    "Ten Twenty Thirty 40 40.9"      #Checking numbers
]

test_cases_2 = [
    "He submitted the acknowledgement.",
    "He submitted all the acknowledgements."
    "Communications."
]

for text in test_cases_2:
    doc = nlp(text)
    print(f"Sentence: \"{text}\"")
    for token in doc:
        # if token.text.lower() == "banking":
            print(f"  Token: {token.text:<10} | POS: {token.pos_:<6} | Tag: {token.tag_:<4} | Lemma (Base Word): {token.lemma_}")
        # print("-----------------------")
        # print("Token: ", {token.text}, "is_digit: ", {token.is_digit}, " ", "like_num: ", {token.like_num})
    print("-" * 80)

Sentence: "He submitted the acknowledgement."
  Token: He         | POS: PRON   | Tag: PRP  | Lemma (Base Word): he
  Token: submitted  | POS: VERB   | Tag: VBD  | Lemma (Base Word): submit
  Token: the        | POS: DET    | Tag: DT   | Lemma (Base Word): the
  Token: acknowledgement | POS: NOUN   | Tag: NN   | Lemma (Base Word): acknowledgement
  Token: .          | POS: PUNCT  | Tag: .    | Lemma (Base Word): .
--------------------------------------------------------------------------------
Sentence: "He submitted all the acknowledgements.Communications."
  Token: He         | POS: PRON   | Tag: PRP  | Lemma (Base Word): he
  Token: submitted  | POS: VERB   | Tag: VBD  | Lemma (Base Word): submit
  Token: all        | POS: DET    | Tag: PDT  | Lemma (Base Word): all
  Token: the        | POS: DET    | Tag: DT   | Lemma (Base Word): the
  Token: acknowledgements | POS: NOUN   | Tag: NNS  | Lemma (Base Word): acknowledgement
  Token: .          | POS: PUNCT  | Tag: .    | Lemma (Base 

---
---

In [ ]:

import os
import pandas as pd


def populate_token_sensitivity(
    input_dir="data/processed",
    combined_filename="combined_dataset.csv",
    tokens_filename="tokens.csv",
    output_filename="tokens_with_sensitivity.csv",
):
    # Construct full file paths
    combined_path = os.path.join(input_dir, combined_filename)
    tokens_path = os.path.join(input_dir, tokens_filename)
    output_path = os.path.join(input_dir, output_filename)

    # Load datasets
    combined_df = pd.read_csv(combined_path)
    tokens_df = pd.read_csv(tokens_path)

    # Validate required columns
    required_combined = {"token", "sensitivity_level"}
    required_tokens = {
        "base_word",
        "is_stopword",
        "is_punct",
        "is_num",
        "is_currency",
        "is_url",
        "is_email",
    }

    missing_combined = required_combined - set(combined_df.columns)
    missing_tokens = required_tokens - set(tokens_df.columns)

    if missing_combined:
        raise ValueError(
            f"Missing columns in combined dataset: {missing_combined}"
        )

    if missing_tokens:
        raise ValueError(
            f"Missing columns in tokens dataset: {missing_tokens}"
        )

    # Normalize dictionary keys and sensitivity labels
    combined_df["token"] = (
        combined_df["token"]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    combined_df["sensitivity_level"] = (
        combined_df["sensitivity_level"]
        .astype("string")
        .str.strip()
        .str.capitalize()
    )

    # Keep only valid dictionary entries
    combined_df = combined_df.dropna(
        subset=["token", "sensitivity_level"]
    )

    valid_levels = {"Low", "Medium", "High"}
    invalid_levels = (
        set(combined_df["sensitivity_level"].unique())
        - valid_levels
    )

    if invalid_levels:
        raise ValueError(
            f"Invalid sensitivity levels: {invalid_levels}"
        )

    # Create token-to-sensitivity lookup
    sensitivity_map = dict(
        zip(
            combined_df["token"],
            combined_df["sensitivity_level"],
        )
    )

    def flag_is_true(value):
        if pd.isna(value):
            return False

        if isinstance(value, bool):
            return value

        return str(value).strip().lower() in {
            "true", "1", "yes"
        }

    def apply_rules(row):
        is_stopword = flag_is_true(row.get("is_stopword"))
        is_punct = flag_is_true(row.get("is_punct"))
        is_num = flag_is_true(row.get("is_num"))
        is_currency = flag_is_true(row.get("is_currency"))
        is_url = flag_is_true(row.get("is_url"))
        is_email = flag_is_true(row.get("is_email"))

        # High sensitivity rules take priority
        if is_num or is_currency or is_url or is_email:
            return "High"

        # Low sensitivity rules
        if is_stopword or is_punct:
            return "Low"

        # Match normalized base word against the dictionary
        base_word = row.get("base_word")

        if pd.notna(base_word):
            base_word = str(base_word).strip().lower()

            if base_word in sensitivity_map:
                return sensitivity_map[base_word]

        # Fail-safe for unmatched terms
        return "High"

    # Populate the sensitivity column
    tokens_df["sensitivity"] = tokens_df.apply(
        apply_rules,
        axis=1,
    )

    # Ensure output directory exists
    os.makedirs(input_dir, exist_ok=True)

    # Save the populated dataset
    tokens_df.to_csv(output_path, index=False)

    # Print summary
    print("Sensitivity population complete.")
    print(f"Input tokens: {len(tokens_df)}")
    print(f"Output saved to: {output_path}")
    print("\nSensitivity distribution:")
    print(
        tokens_df["sensitivity"]
        .value_counts(dropna=False)
        .to_string()
    )

    return tokens_df


if __name__ == "__main__":
    populate_token_sensitivity()


Sensitivity population complete.
Input tokens: 657
Output saved to: data/processed/tokens_with_sensitivity.csv

Sensitivity distribution:
sensitivity
High      303
Low       238
Medium    116


# 2 - Cryptography Section


a) Installing the necessary libraries.

In [ ]:
pip install pymupdf pandas cryptography

b) Generating two keys: Private and Public for RSA Encryption

In [ ]:

from pathlib import Path

from cryptography.hazmat.primitives import serialization
from cryptography.hazmat.primitives.asymmetric import rsa


def generate_key_pair(output_dir="keys"):
    """Generate and save an RSA public/private key pair."""

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    private_key = rsa.generate_private_key(
        public_exponent=65537,
        key_size=2048
    )

    public_key = private_key.public_key()

    private_path = output_dir / "private_key.pem"
    public_path = output_dir / "public_key.pem"

    private_path.write_bytes(
        private_key.private_bytes(
            encoding=serialization.Encoding.PEM,
            format=serialization.PrivateFormat.PKCS8,
            encryption_algorithm=serialization.NoEncryption()
        )
    )

    public_path.write_bytes(
        public_key.public_bytes(
            encoding=serialization.Encoding.PEM,
            format=serialization.PublicFormat.SubjectPublicKeyInfo
        )
    )

    print(f"Public key saved to: {public_path}")
    print(f"Private key saved to: {private_path}")

    return str(public_path), str(private_path)

if __name__ == "__main__":
    generate_key_pair()

Public key saved to: keys/public_key.pem
Private key saved to: keys/private_key.pem


c) Shared cryptographic functions.

Keeping them separate prevents us from repeating key derivation and authentication logic in the encryption and decryption files.

In [ ]:

import base64
import json

from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.kdf.hkdf import HKDF


# Output key lengths are measured in bytes.
SENSITIVITY_KEY_LENGTHS = {
    "Low": 16,       # AES-128
    "Medium": 24,    # AES-192
    "High": 32       # AES-256
}

SENSITIVITY_RANK = {
    "Low": 1,
    "Medium": 2,
    "High": 3
}


def encode_bytes(data):
    """Convert bytes to a JSON-compatible Base64 string."""
    return base64.b64encode(data).decode("utf-8")


def decode_bytes(data):
    """Convert a Base64 string back to bytes."""
    return base64.b64decode(data)


def derive_aes_key(master_key, salt, sensitivity):
    """Derive the AES key appropriate for a sensitivity level."""

    if sensitivity not in SENSITIVITY_KEY_LENGTHS:
        raise ValueError(
            f"Unsupported sensitivity level: {sensitivity}"
        )

    key_length = SENSITIVITY_KEY_LENGTHS[sensitivity]

    return HKDF(
        algorithm=hashes.SHA256(),
        length=key_length,
        salt=salt,
        info=f"selective-pdf:{sensitivity}".encode("utf-8")
    ).derive(master_key)


def make_aad(record_id, page_number, sensitivity):
    """
    Create authenticated metadata for AES-GCM.

    The same values must be used during encryption and decryption.
    """
    metadata = {
        "record_id": str(record_id),
        "page_number": int(page_number),
        "sensitivity": sensitivity
    }

    return json.dumps(
        metadata,
        sort_keys=True,
        separators=(",", ":")
    ).encode("utf-8")


d) Encrypting the uploaded document.

Revision: Ensuring the encrypted payload has text formatting information to be used during decryption.

In [ ]:

import json
from collections import defaultdict
from pathlib import Path

import fitz
import pandas as pd

from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import padding
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from cryptography.hazmat.primitives.kdf.hkdf import HKDF
from cryptography.hazmat.primitives import hashes as crypto_hashes

# Not required in collab since previous cell stores these values. Must when copied to separete file.
# from crypto_utils import (
#     SENSITIVITY_RANK,
#     derive_aes_key,
#     encode_bytes,
#     make_aad
# )

PAYLOAD_NAME = "encrypted_token_payload.json"


def load_public_key(public_key_path):
    """Load the recipient's RSA public key."""

    with open(public_key_path, "rb") as file:
        return serialization.load_pem_public_key(file.read())


def load_layout(layout_json_path):
    """Load the original page and word layout generated by NLP."""

    with open(layout_json_path, "r", encoding="utf-8") as file:
        return json.load(file)


def layout_word_key(page_number, block_no, line_no, word_no):
    """Build a key shared by the NLP CSV and layout JSON."""

    return (
        int(page_number),
        int(block_no),
        int(line_no),
        int(word_no)
    )


def build_sensitivity_map(tokens_df):
    """
    Map each original PDF word to its sensitivity level.

    A PDF word may correspond to multiple spaCy tokens.
    If their sensitivity labels differ, use the highest label.
    """

    required_columns = {
        "page_number",
        "block_no",
        "line_no",
        "word_no",
        "sensitivity",
        "token"
    }

    missing = required_columns - set(tokens_df.columns)
    if missing:
        raise ValueError(
            f"Missing CSV columns: {sorted(missing)}"
        )

    valid_levels = {"Low", "Medium", "High"}
    sensitivity_map = {}

    for _, row in tokens_df.iterrows():
        values = [
            row["page_number"],
            row["block_no"],
            row["line_no"],
            row["word_no"]
        ]

        if any(pd.isna(value) for value in values):
            continue

        sensitivity = str(row["sensitivity"]).strip()
        token = str(row["token"]).strip()

        if sensitivity not in valid_levels:
            raise ValueError(
                f"Invalid sensitivity label for {token}: {sensitivity!r}"
            )

        key = layout_word_key(*values)
        current = sensitivity_map.get(key)

        if (
            current is None
            or SENSITIVITY_RANK[sensitivity]
            > SENSITIVITY_RANK[current]
        ):
            sensitivity_map[key] = sensitivity

    return sensitivity_map


def wrap_session_key(master_key, public_key):
    """Encrypt the master session key with RSA-OAEP."""

    return public_key.encrypt(
        master_key,
        padding.OAEP(
            mgf=padding.MGF1(algorithm=hashes.SHA256()),
            algorithm=hashes.SHA256(),
            label=None
        )
    )


def encrypt_document(
    pdf_path,
    tokens_csv_path,
    layout_json_path,
    public_key_path,
    output_pdf_path="outputs/encrypted_document.pdf",
    output_session_key_path="outputs/encrypted_session_key.bin"
):
    """
    Encrypt PDF words according to their ML sensitivity labels.

    The original PDF is redacted, and encrypted token records are
    embedded in the resulting PDF. The master session key is
    separately encrypted using the recipient's RSA public key.
    """

    pdf_path = Path(pdf_path)
    output_pdf_path = Path(output_pdf_path)
    output_session_key_path = Path(output_session_key_path)

    if not pdf_path.exists():
        raise FileNotFoundError(f"PDF not found: {pdf_path}")

    tokens_df = pd.read_csv(tokens_csv_path)
    layout_pages = load_layout(layout_json_path)
    public_key = load_public_key(public_key_path)

    sensitivity_map = build_sensitivity_map(tokens_df)

    layout_keys = {
        layout_word_key(
            page["page_number"],
            word["block_no"],
            word["line_no"],
            word["word_no"]
        )
        for page in layout_pages
        for word in page.get("words", [])
        if word.get("word")
    }

    mapped_keys = set(sensitivity_map)

    missing_keys = layout_keys - mapped_keys
    extra_keys = mapped_keys - layout_keys

    print(f"PDF layout words: {len(layout_keys)}")
    print(f"Words with sensitivity labels: "
          f"{len(layout_keys & mapped_keys)}")
    print(f"Words missing labels: {len(missing_keys)}")
    print(f"Unused CSV word keys: {len(extra_keys)}")

    if missing_keys:
        print("Sample missing layout keys:")
        for key in list(sorted(missing_keys))[:20]:
            print(key)


    # One random 256-bit master key for this document.
    master_key = AESGCM.generate_key(bit_length=256)

    # The salt is stored with the encrypted payload, not the plaintext.
    salt = __import__("secrets").token_bytes(16)

    encrypted_records = []
    redactions_by_page = defaultdict(list)

    with fitz.open(pdf_path) as pdf:
        if len(pdf) != len(layout_pages):
            raise ValueError(
                "The PDF page count does not match pdf_layout.json."
            )

        for layout_page in layout_pages:
            page_number = int(layout_page["page_number"])
            page_index = page_number - 1

            if not (0 <= page_index < len(pdf)):
                raise ValueError(
                    f"Invalid page number in layout: {page_number}"
                )

            page = pdf[page_index]

            # Validate that the layout corresponds to this PDF.
            if (
                abs(page.rect.width - float(
                    layout_page["page_width"]
                )) > 2
                or abs(page.rect.height - float(
                    layout_page["page_height"]
                )) > 2
            ):
                raise ValueError(
                    f"Page dimensions mismatch on page {page_number}."
                )

            for word in layout_page.get("words", []):
                word_text = word.get("word", "")

                if not word_text:
                    continue

                key = layout_word_key(
                    page_number,
                    word["block_no"],
                    word["line_no"],
                    word["word_no"]
                )

                sensitivity = sensitivity_map.get(key)

                if sensitivity is None:
                  print(
                      f"WARNING: No sensitivity label for "
                      f"page={page_number}, "
                      f"word={word_text!r}. "
                      f"Defaulting to High."
                  )
                  sensitivity = "High"

                x0, y0, x1, y1 = map(float, word["bbox"])

                if x1 <= x0 or y1 <= y0:
                    continue

                # Encrypt the original PDF word, not its lemma.
                record_id = (
                    f"P{page_number:03d}_"
                    f"B{word['block_no']}_"
                    f"L{word['line_no']}_"
                    f"W{word['word_no']}"
                )

                nonce = __import__("secrets").token_bytes(12)

                aes_key = derive_aes_key(
                    master_key,
                    salt,
                    sensitivity
                )

                aad = make_aad(
                    record_id,
                    page_number,
                    sensitivity
                )

                ciphertext = AESGCM(aes_key).encrypt(
                    nonce,
                    word_text.encode("utf-8"),
                    aad
                )

                encrypted_records.append({
                    "record_id": record_id,
                    "page_number": page_number,
                    "sensitivity": sensitivity,
                    "bbox": [x0, y0, x1, y1],
                    "block_no": word["block_no"],
                    "line_no": word["line_no"],
                    "word_no": word["word_no"],
                    "nonce": encode_bytes(nonce),
                    "ciphertext": encode_bytes(ciphertext),
                    "font": word.get("font"),
                    "font_size": word.get("font_size"),
                    "color": word.get("color"),
                    "flags": word.get("flags"),
                    "bold": word.get("bold"),
                    "italic": word.get("italic"),
                })

                # Redact this original word from the visible PDF.
                redactions_by_page[page_index].append(
                    fitz.Rect(x0, y0, x1, y1)
                )

        if not encrypted_records:
            raise ValueError(
                "No PDF words matched the sensitivity CSV. "
                "Check page, block, line and word numbers."
            )

        # Add redaction annotations before applying them.
        for page_index, rectangles in redactions_by_page.items():
            page = pdf[page_index]

            for rect in rectangles:
                page.add_redact_annot(
                    rect,
                    fill=(1, 1, 1),
                    cross_out=False
                )

        # Apply redactions permanently to the PDF's text content.
        for page_index in redactions_by_page:
            pdf[page_index].apply_redactions(
                images=0,
                graphics=0,
                text=0
            )

        payload = {
            "format_version": 1,
            "salt": encode_bytes(salt),
            "records": encrypted_records
        }

        payload_bytes = json.dumps(
            payload,
            separators=(",", ":")
        ).encode("utf-8")

        output_pdf_path.parent.mkdir(
            parents=True,
            exist_ok=True
        )
        output_session_key_path.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        # Store ciphertext and reconstruction coordinates inside the PDF.
        pdf.embfile_add(
            PAYLOAD_NAME,
            payload_bytes,
            filename=PAYLOAD_NAME,
            desc="Encrypted token payload"
        )

        pdf.save(
            str(output_pdf_path),
            garbage=4,
            deflate=True
        )

    # Protect the master session key using RSA-OAEP.
    encrypted_session_key = wrap_session_key(
        master_key,
        public_key
    )

    output_session_key_path.write_bytes(
        encrypted_session_key
    )

    print(f"Encrypted PDF: {output_pdf_path}")
    print(f"Encrypted session key: {output_session_key_path}")
    print(f"Encrypted words: {len(encrypted_records)}")

    return (
        str(output_pdf_path),
        str(output_session_key_path)
    )


if __name__ == "__main__":
    encrypt_document(
        pdf_path="/content/Educational_Data_Document.pdf",
        tokens_csv_path="/content/data/processed/tokens_with_sensitivity.csv",
        layout_json_path="/content/data/processed/pdf_layout.json",
        public_key_path="/content/keys/public_key.pem"
    )


PDF layout words: 585
Words with sensitivity labels: 585
Words missing labels: 0
Unused CSV word keys: 0
Encrypted PDF: outputs/encrypted_document.pdf
Encrypted session key: outputs/encrypted_session_key.bin
Encrypted words: 585


e) Decrypting the uploaded document

Revision: Using the newly avaiable text formatting data to decrypt the document as closely to the original as possible.

In [ ]:
from pathlib import Path

import fitz

from cryptography.hazmat.primitives import serialization
from cryptography.hazmat.primitives.asymmetric import padding
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.ciphers.aead import AESGCM

# from crypto_utils import (
#     decode_bytes,
#     derive_aes_key,
#     make_aad
# )

# from encrypt_document import PAYLOAD_NAME


def load_private_key(private_key_path):
    """Load the recipient's RSA private key."""

    with open(private_key_path, "rb") as file:
        return serialization.load_pem_private_key(
            file.read(),
            password=None
        )


def unwrap_session_key(encrypted_session_key_path, private_key):
    """Decrypt the master session key using RSA-OAEP."""

    encrypted_session_key = Path(
        encrypted_session_key_path
    ).read_bytes()

    return private_key.decrypt(
        encrypted_session_key,
        padding.OAEP(
            mgf=padding.MGF1(algorithm=hashes.SHA256()),
            algorithm=hashes.SHA256(),
            label=None
        )
    )


def read_encrypted_payload(pdf):
    """Read the encrypted token metadata embedded in the PDF."""

    if PAYLOAD_NAME not in pdf.embfile_names():
        raise ValueError(
            "The PDF does not contain the expected encrypted payload."
        )

    import json

    payload_bytes = pdf.embfile_get(PAYLOAD_NAME)
    return json.loads(payload_bytes.decode("utf-8"))


def decrypt_record(record, master_key, salt):
    """Authenticate and decrypt one encrypted PDF word."""

    sensitivity = record["sensitivity"]

    aes_key = derive_aes_key(
        master_key,
        salt,
        sensitivity
    )

    nonce = decode_bytes(record["nonce"])
    ciphertext = decode_bytes(record["ciphertext"])

    aad = make_aad(
        record["record_id"],
        record["page_number"],
        sensitivity
    )

    # Invalid authentication raises an exception.
    plaintext_bytes = AESGCM(aes_key).decrypt(
        nonce,
        ciphertext,
        aad
    )

    return plaintext_bytes.decode("utf-8")



def restore_word(page, record, plaintext):
    """Restore a decrypted word using saved formatting metadata."""

    x0, y0, x1, y1 = map(float, record["bbox"])
    height = y1 - y0

    if height <= 0:
        return

    # Use the original font size when available.
    font_size = record.get("font_size")

    if font_size is None:
        font_size = max(4.0, height * 0.85)
    else:
        font_size = float(font_size)  * 0.90

    # Approximate the original bold/italic style with built-in fonts.
    bold = record.get("bold") is True
    italic = record.get("italic") is True

    if bold and italic:
        font_name = "hebi"
    elif bold:
        font_name = "hebo"
    elif italic:
        font_name = "heit"
    else:
        font_name = "helv"

    # Convert the saved integer RGB color to PyMuPDF's 0-1 RGB format.
    original_color = record.get("color")

    if original_color is not None:
        original_color = int(original_color)

        color = (
            ((original_color >> 16) & 255) / 255.0,
            ((original_color >> 8) & 255) / 255.0,
            (original_color & 255) / 255.0
        )
    else:
        color = (0, 0, 0)

    # Approximate the original baseline from its bounding box.
    baseline_y = y1 - max(1.0, height * 0.15)

    page.insert_text(
        (x0, baseline_y),
        plaintext,
        fontsize=font_size,
        fontname=font_name,
        color=color,
        overlay=True
    )



def decrypt_document(
    encrypted_pdf_path,
    encrypted_session_key_path,
    private_key_path,
    output_pdf_path="outputs/decrypted_document.pdf"
):
    """
    Decrypt an encrypted PDF and restore its text.

    Inputs:
      1. Encrypted PDF
      2. Encrypted session key
      3. Recipient's private RSA key
    """

    encrypted_pdf_path = Path(encrypted_pdf_path)
    output_pdf_path = Path(output_pdf_path)

    if not encrypted_pdf_path.exists():
        raise FileNotFoundError(
            f"Encrypted PDF not found: {encrypted_pdf_path}"
        )

    private_key = load_private_key(private_key_path)

    master_key = unwrap_session_key(
        encrypted_session_key_path,
        private_key
    )

    output_pdf_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    with fitz.open(encrypted_pdf_path) as pdf:
        payload = read_encrypted_payload(pdf)
        salt = decode_bytes(payload["salt"])

        records = payload["records"]

        for record in records:
            page_number = int(record["page_number"])
            page_index = page_number - 1

            if not (0 <= page_index < len(pdf)):
                raise ValueError(
                    f"Invalid page number: {page_number}"
                )

            plaintext = decrypt_record(
                record,
                master_key,
                salt
            )

            restore_word(
                pdf[page_index],
                record,
                plaintext
            )

        # The decrypted text is now restored, so remove the embedded
        # ciphertext payload from the decrypted output.
        pdf.embfile_del(PAYLOAD_NAME)

        pdf.save(
            str(output_pdf_path),
            garbage=4,
            deflate=True
        )

    print(f"Decrypted PDF: {output_pdf_path}")
    print(f"Restored words: {len(records)}")

    return str(output_pdf_path)


if __name__ == "__main__":
    decrypt_document(
        encrypted_pdf_path="outputs/encrypted_document.pdf",
        encrypted_session_key_path="outputs/encrypted_session_key.bin",
        private_key_path="keys/private_key.pem"
    )


Decrypted PDF: outputs/decrypted_document.pdf
Restored words: 585
